# Certification experiment for cascaded tanks

This notebook runs the paper experiment on Google Colab:

- clones the project repository;
- installs Python dependencies;
- uploads and activates a Gurobi license file;
- runs `paper_experiment.py` for `N = {1,2,5,10,20}`;
- shows the resulting table;
- packages `.dat` and `.tex` files for download.

## 1. Configure repository

Replace `REPO_URL` with your GitHub repository URL before running.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/nn_cert.git"
BRANCH = "main"

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL to your real GitHub repository URL before continuing.")

## 2. Install dependencies

In [ ]:
!pip -q install gurobipy numpy scipy matplotlib torch

## 3. Clone repository

In [ ]:
import os
import shutil

PROJECT_DIR = "/content/nn_cert"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls -la

## 4. Upload Gurobi license

Upload your `gurobi.lic` file. If your file has another name, the cell below will rename the uploaded file to `/content/gurobi.lic` and set `GRB_LICENSE_FILE`.

In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one Gurobi license file.")

uploaded_name = next(iter(uploaded.keys()))
license_path = "/content/gurobi.lic"
shutil.copy(uploaded_name, license_path)
os.environ["GRB_LICENSE_FILE"] = license_path
print("Using Gurobi license at", license_path)

## 5. Test Gurobi license

In [ ]:
import gurobipy as gp
from gurobipy import GRB

m = gp.Model("license_test")
m.Params.OutputFlag = 1
x = m.addVar(lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x")
m.setObjective(x, GRB.MAXIMIZE)
m.optimize()
print("status:", m.Status, "objective:", m.ObjVal if m.SolCount > 0 else None)

## 6. Run paper experiment

This trains the networks and solves the MILPs to optimality for `N = [1, 2, 5, 10, 20]`. It may take a while.

In [ ]:
!python paper_experiment.py

## 7. Show certification table

In [ ]:
import pandas as pd

table_path = "paper_results/certification_table.dat"
table = pd.read_csv(
    table_path,
    comment="#",
    sep=r"\s+",
    names=["N", "objective", "gap_percent", "solving_time_seconds"],
)
table

## 8. Preview one trajectory

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_TO_PREVIEW = 10
data = np.loadtxt(f"paper_results/output_curve_N{N_TO_PREVIEW}.dat", comments="#")

k = data[:, 0]
y_ref = data[:, 1]
y_cand = data[:, 2]
err = data[:, 3]

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(k, y_ref, marker="o", label="reference")
axes[0].plot(k, y_cand, marker="s", label="candidate")
axes[0].grid(True)
axes[0].legend()
axes[0].set_ylabel("y")

axes[1].plot(k, err, marker="d", color="tab:red", label="absolute error")
axes[1].grid(True)
axes[1].legend()
axes[1].set_xlabel("k")
axes[1].set_ylabel("|error|")
plt.tight_layout()
plt.show()

## 9. Package and download results

In [ ]:
!zip -r paper_results.zip paper_results
files.download("paper_results.zip")